In [11]:
!pip install fasttext transformers mlflow spacy pympler

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.8/165.8 kB 6.1 MB/s eta 0:00:00


In [2]:
!python -m spacy download ru_core_news_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 66.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [15]:
from abc import ABC, abstractmethod
from enum import Enum

import fasttext
import fasttext.util
import mlflow.transformers
import numpy as np
import spacy
import torch
import time
from statistics import mean, stdev
from pympler import asizeof
import gc

In [4]:
MLFLOW_TRACKING_URI = "http://70.34.242.179"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [5]:
class ToxicityType(str, Enum):
    INSULT = "INSULT"
    NORMAL = "NORMAL"
    OBSCENITY = "OBSCENITY"
    THREAT = "THREAT"


class TextTokenizer(ABC):

    @abstractmethod
    def encode(self, texts: list[str]) -> tuple[torch.Tensor, torch.Tensor]:
        pass

    @abstractmethod
    def decode(self, id: int) -> str:
        pass

class FasttextTokenizer(TextTokenizer):

    FASTTEXT_MODEL_FILE_NAME = "cc.ru.300.bin"

    def __init__(self):
        fasttext.util.download_model("ru", if_exists="ignore")
        self.fasttext_model = fasttext.load_model(self.FASTTEXT_MODEL_FILE_NAME)


    def encode(self, texts: list[str], max_len=30):
        embeddings = []
        zero_vector = torch.zeros(300)
        mask = torch.fill_(torch.zeros(max_len), 1)
        for text in texts:
            words = text.split()[:max_len]
            vectors = [
                torch.from_numpy(self.fasttext_model.get_word_vector(word))
                for word in words
            ]

            if len(vectors) < max_len:
                vectors += [zero_vector] * (max_len - len(vectors))

            embeddings.append(torch.vstack(vectors).unsqueeze(0))
        return torch.vstack(embeddings), mask

    def decode(self, id: int) -> str:
        raise NotImplemented()

class BaseToxicityPredictor(ABC):

    __CLASS_LABELS = ["NORMAL", "INSULT", "THREAT", "OBSCENITY"]
    __CLASS_IDS = [0, 1, 2, 3]
    __CLASS_MAPPING = {
        0: ToxicityType.NORMAL,
        1: ToxicityType.INSULT,
        2: ToxicityType.THREAT,
        3: ToxicityType.OBSCENITY
    }

    def predict(self, text: str) -> str:
        label_id = self.predict_proba(text).argmax()
        return self.__CLASS_MAPPING[label_id]

    @abstractmethod
    def predict_proba(self, text: str) -> np.ndarray:
        pass


class BertToxicityPredictor(BaseToxicityPredictor):

    def __init__(self, model_name, model_alias="final"):
        self.model_name = model_name
        self.model_alias = model_alias
        self.model = mlflow.transformers.load_model(
            model_uri=f"models:/{self.model_name}@{self.model_alias}",
            return_type="pipeline",
            map_location=torch.device("cpu")
        )

    def predict_proba(self, text: str) -> np.ndarray:
        preds = self.model([text], top_k=None)[0]

        preds = sorted(preds, key=lambda item: item["label"])

        return np.array([pred["score"] for pred in preds], dtype=np.float32)


class LSTMToxicityPredictor(BaseToxicityPredictor):

    def __init__(self, model_name="improved_LSTM", model_alias="final"):
        self.model_name = model_name
        self.model_alias = model_alias
        self.tokenizer = FasttextTokenizer()
        self.model = mlflow.pytorch.load_model(
            f"models:/{self.model_name}@{self.model_alias}",
            map_location=torch.device("cpu")
        )

    def predict_proba(self, text: str) -> np.ndarray:
        input, _ = self.tokenizer.encode([text], max_len=len(text))
        probas = self.model(input)
        return probas.detach().numpy()[0]


class LogRegToxicityPredictor(BaseToxicityPredictor):

    __ALLOWED_PUNCT = {'!', '?'}

    def __init__(self, model_name="baseline_logreg_bow", model_alias="final"):
        self.model_name = model_name
        self.model_alias = model_alias
        self.nlp = spacy.load("ru_core_news_md")
        self.model = mlflow.sklearn.load_model(
            f"models:/{self.model_name}@{self.model_alias}"
        )

    def __lemmatize(self, text: str):
        cleaned = []
        for token in self.nlp(text):
            if token.is_stop:
                continue
            if token.is_alpha:
                lemma = token.lemma_
                if len(lemma) < 3 or len(lemma) > 30:
                    continue
                else:
                    cleaned.append(lemma)
            elif token.is_punct:
                if token.text in self.__ALLOWED_PUNCT:
                    cleaned.append(token.text)
            else:
                cleaned.append(token.text)
        return ' '.join(cleaned)

    def predict_proba(self, text: str) -> np.ndarray:
        lemmatized = self.__lemmatize(text)
        return self.model.predict_proba([lemmatized])[0]



In [6]:
big_bert_predictor = BertToxicityPredictor("BERT_clf_dropout_DeepPavlov_rubert-base-cased-conversational")
tiny_bert_predictor = BertToxicityPredictor("BERT_clf_dropout_cointegrated_rubert-tiny2")
lstm_predictor = LSTMToxicityPredictor()
logreg_predictor = LogRegToxicityPredictor()

2026/06/02 20:54:52 INFO mlflow.transformers: 'models:/BERT_clf_dropout_DeepPavlov_rubert-base-cased-conversational@final' resolved as 'mlflow-artifacts:/8/models/m-b329ff674a404325a8c090c6a9bfc1cf/artifacts'


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


2026/06/02 20:55:35 INFO mlflow.transformers: 'models:/BERT_clf_dropout_cointegrated_rubert-tiny2@final' resolved as 'mlflow-artifacts:/7/models/m-8ed1ad0cab57410ca0a79883256aaf13/artifacts'


Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

2026/06/02 21:08:28 WARNING mlflow.pytorch: Stored model version '2.10.0+cu128' does not match installed PyTorch version '2.11.0+cpu'


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator CountVectorizer from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.7.2 when using version 1.6.1. This might lead to breaking code or i

In [7]:
TEXT = "эмилия кравец хочет сдохнуть молодой. ха ха. :d"

def benchmark(predictor, runs=100, warmup=5):
    for _ in range(warmup):
        predictor.predict(TEXT)

    times_ms = []

    for _ in range(runs):
        start = time.perf_counter()
        predictor.predict(TEXT)
        end = time.perf_counter()

        times_ms.append((end - start) * 1000)

    avg = mean(times_ms)

    result = {
        "runs": runs,
        "average_ms": avg,
        "min_ms": min(times_ms),
        "max_ms": max(times_ms),
    }

    if runs > 1:
        result["stdev_ms"] = stdev(times_ms)

    return result



In [16]:
gc.collect()

1227

In [17]:
stats = benchmark(big_bert_predictor, runs=50, warmup=5)
print(f"Среднее время [BERT conversational]: {stats['average_ms']:.3f} ms")
print(f"Занимаемое место: {asizeof.asizeof(big_bert_predictor) / 1024 ** 2:.3f} MB")

stats = benchmark(tiny_bert_predictor, runs=50, warmup=5)
print(f"Среднее время [BERT tiny]: {stats['average_ms']:.3f} ms")
print(f"Занимаемое место: {asizeof.asizeof(tiny_bert_predictor) / 1024 ** 2:.3f} MB")

stats = benchmark(lstm_predictor, runs=50, warmup=5)
print(f"Среднее время [LSTM]: {stats['average_ms']:.3f} ms")
print(f"Занимаемое место: {asizeof.asizeof(lstm_predictor) / 1024 ** 2:.3f} MB")

stats = benchmark(logreg_predictor, runs=50, warmup=5)
print(f"Среднее время [LogReg]: {stats['average_ms']:.3f} ms")
print(f"Занимаемое место: {asizeof.asizeof(logreg_predictor) / 1024 ** 2:.3f} MB")

Среднее время [BERT conversational]: 214.987 ms
Занимаемое место: 14.290 MB
Среднее время [BERT tiny]: 12.067 ms
Занимаемое место: 10.097 MB
Среднее время [LSTM]: 54.371 ms
Занимаемое место: 0.040 MB
Среднее время [LogReg]: 24.055 ms
Занимаемое место: 46.954 MB
